In [122]:
import sys
sys.path.insert(1, '../../../scripts/')
from preprocess import preprocess
from preprocess import correct_inputs 

from utils import functions as func
from utils import parameters as params
from utils import metabolites as metab

from tqdm import tqdm
import copy
import pickle
import sympy

import numpy as np
import pandas as pd
import pickle

lp_path = '/data2/hratch/human_me/other/test_lp/'

In [141]:
jabba = True
counter = 3
base = 0
mu_val = 1e-9
fn = '/data2/hratch/human_me/other/test_lp/S_matrix.h5'

In [4]:
# from expression import build_me_model
# tme, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = True, 
#                                             dummy_protein = False)

# if jabba:
#     for r in tme.reactions:
#         if ('EX_' in r.id and r.compartments == {'b'} and r.bounds == (float('-inf'), float('inf'))):
#             r._lower_bound = -1000
#             r._upper_bound = 1000
    
    
# with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
#     tme = pickle.load(handle)

# with open(lp_path + 'notworking_version.pickle', 'rb') as handle:
#     tme = pickle.load(handle)

In [142]:
with open(lp_path + 'working_version_' + str(base) + '.pickle', 'rb') as handle:
    tme0 = pickle.load(handle)

with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
    tme1 = pickle.load(handle)
    


In [ ]:
sln0, stat0, _ = tme0.solve_lp(mu_val = mu_val)    
sln1, stat1, _ = tme1.solve_lp(mu_val = mu_val)    

In [5]:
import pandas as pd
res_df = pd.DataFrame(data = {'reactions': [r.id for r in tme0.reactions]})
res_df['fluxes'] = res_df.reactions.apply(lambda x: sln0[tme0.reactions.index(x)])
res_df.set_index(res_df.reactions, drop = True, inplace = True)
biom = res_df.loc[[i for i in res_df.index if 'biomass' in i],:]

biom.sort_values(by = 'fluxes', ascending = False)

,reactions,fluxes
reactions,,
biomass_dilution,biomass_dilution,1.000000e-09
DNA_biomass_formation,DNA_biomass_formation,1.000000e-09
carbohydrate_biomass_formation,carbohydrate_biomass_formation,1.000000e-09
lipid_biomass_formation,lipid_biomass_formation,1.000000e-09
mRNA_biomass_to_biomass,mRNA_biomass_to_biomass,4.899296e-10
protein_biomass_to_biomass,protein_biomass_to_biomass,3.280437e-10
lipid_biomass_to_biomass,lipid_biomass_to_biomass,9.700000e-11
carbohydrate_biomass_to_biomass,carbohydrate_biomass_to_biomass,7.100000e-11
DNA_biomass_to_biomass,DNA_biomass_to_biomass,1.400000e-11


In [ ]:
# with open(lp_path + 'working_version' + str(counter) + '.pickle', 'rb') as handle:
#     tme = pickle.load(handle)

In [ ]:
sln, stat, _ = tme.solve_lp(mu_val = mu_val)

S = tme.create_stoichiometric_matrix(mu_val = 1, inplace = False, array_type = 'pandas')
res = pd.DataFrame(data = {'reaction_fluxes': sln[:len(tme.reactions)]})
res.index = [r.id for r in tme.reactions]

if stat == 0:
    S.to_hdf(fn, key = str(counter), mode = 'a')
    print('Last saved file: {}'.format(counter))
else:
    infeasible_reactions = tme.infeasible_reactions(mu_val = mu_val, sln = sln, stat = stat)
    print('Model did not solve')
    
res.loc[[i for i in res.index if 'biomass' in i],:].sort_values(by = 'reaction_fluxes', ascending = False)

In [195]:
def save_me_model(me_model, counter):
    print('Success, please update git')
#     lp_path = '/data2/hratch/human_me/test_lp/'
#     me_model.pickle(lp_path + 'working_version_' + str(counter) + '.pickle')

def get_changes(S_1, S_0):
    mismatch = np.argwhere(np.not_equal(S_0.values, S_1.values))
    am = S_1.index.tolist()
    mm = {m: {'id': am[m]} for m in sorted(set([t[0] for t in mismatch]))}
    for m in mm:
        mm[m]['reactions'] = sorted(set([t[1] for t in mismatch if t[0] == m]))

    am, rm = S_1.index.tolist(), S_1.columns.tolist()
    mm_2 = {m: {'id': am[m]} for m in sorted(set([t[0] for t in mismatch]))}
    for m in mm_2:
        mm_2[m]['reactions'] = sorted(set(['_'.join(rm[t[1]].split('_')[1:]) if 'HGNC' in rm[t[1]] else rm[t[1]] for t in mismatch if t[0] == m]))
    
    return mismatch, mm, mm_2

In [4]:
# S_0 = pd.read_hdf(fn, key = str(base))
# S_1 = pd.read_hdf(fn, key = str(counter))

In [143]:
S_1 = tme1.create_stoichiometric_matrix(mu_val = 1, inplace = False, array_type = 'pandas')
S_0 = tme0.create_stoichiometric_matrix(mu_val = 1, inplace = False, array_type = 'pandas')

In [196]:
mismatch, mm, mm_2 =  get_changes(S_1, S_0)

In [145]:
set(S_1.index).difference(S_0.index)

{'191091342_complex_n'}

In [146]:
set(S_0.index).difference(S_1.index)

{'125250296_complex_n'}

In [147]:
set(S_0.columns).difference(S_1.columns)

set()

In [128]:
mapper = {
    'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_COMPLEX_COPI_RETROtr': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_COMPLEX_COPI_RETROtr',
    'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_COMPLEX_DEUBIQUITINATIONc': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_COMPLEX_DEUBIQUITINATIONc',
    'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_COMPLEX_FORMATIONg': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_COMPLEX_FORMATIONg', 
    'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_COMPLEX_POLYUBIQUITINATIONc': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_COMPLEX_POLYUBIQUITINATIONc',
    'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_COMPLEX_PROTEASOMAL_DEGRADATIONc': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_COMPLEX_PROTEASOMAL_DEGRADATIONc',
    'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_COMPLEX_RETROTRANSLOCATION': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_COMPLEX_RETROTRANSLOCATION',
    'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_COMPLEX_UNFOLDr': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_COMPLEX_UNFOLDr'
   
}

S_1.columns = [col if col not in mapper else mapper[col] for col in S_1.columns.tolist()]

# mapper = {
#     '266375715_complex_n': '125250296_complex_n', 
#     'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_COMPLEX_enzyme_deg_proxy': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_COMPLEX_enzyme_deg_proxy',
#     'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_complex_c': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_complex_c',
#     'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_complex_g': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_complex_g',
#     'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_complex_r': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_complex_r',
#     'Clathrin_IMPORTtl_Clathrin_IMPORTtpm_polyub_complex_c': 'Clathrin_IMPORTtpm_Clathrin_IMPORTtl_polyub_complex_c',
#     'unfolded_Clathrin_IMPORTtl_Clathrin_IMPORTtpm_complex_r': 'unfolded_Clathrin_IMPORTtpm_Clathrin_IMPORTtl_complex_r'
# }
mapper = {'191091342_complex_n': '125250296_complex_n'}
S_1.index = [col if col not in mapper else mapper[col] for col in S_1.index.tolist()]


S_1 = S_1.loc[S_0.index.tolist(), S_0.columns.tolist()]

In [148]:
mapper = {'191091342_complex_n': '125250296_complex_n'}
S_1.index = [col if col not in mapper else mapper[col] for col in S_1.index.tolist()]


S_1 = S_1.loc[S_0.index.tolist(), S_0.columns.tolist()]

In [149]:
if S_1.shape != S_0.shape:
    print('Dimensions are not the same')
    indeces = False
if len(set(S_1.columns).difference(S_0.columns)) > 0:
    print('Columns are not the same')
    indeces = False
if len(set(S_1.index).difference(S_0.index)) > 0:
    print('Rows are not the same')
    indeces = False

In [5]:
# S_0 = pd.read_hdf(fn, key = str(base))

if not S_0.equals(S_1):
    indeces = True
    if indeces:
        if S_1.shape != S_0.shape:
            print('Dimensions are not the same')
            indeces = False
        if len(set(S_1.columns).difference(S_0.columns)) > 0:
            print('Columns are not the same')
            indeces = False
        if len(set(S_1.index).difference(S_0.index)) > 0:
            print('Rows are not the same')
            indeces = False
    if indeces:
        S_1 = S_1.loc[S_0.index, S_0.columns]
        if not S_0.equals(S_1):
            mismatch, mm, mm_2 = get_changes(S_1, S_0)
            print('Dataframes are not equal due to stoichiometric values mismatch, will not save model')
        else:
            save_me_model(tme, counter)
    else:
        print('Dataframes are not equal due to column/row label mismatch, will not save model')
else:
    save_me_model(tme, counter)

Columns are not the same
Rows are not the same
Dataframes are not equal due to column/row label mismatch, will not save model


In [ ]:
# idx = S_1.index.tolist()
# for i in range(len(idx)):
#     val = idx[i]
#     if 'folded' in val and 'polyub' in val:
#         idx[i] = val.replace('protein_polyub', 'protein_' + val[-1] + '_polyub')
# S_1.index = idx

# idx = set(S_0.index).intersection(S_1.index)
# col = set(S_0.columns).intersection(S_1.columns)
# S_0 = S_0.loc[idx, col]
# S_1 = S_1.loc[idx, col]

print(set(S_1.index).difference(S_0.index))
print(set(S_0.index).difference(S_1.index))

for tr, v in dict(zip(list(set(S_1.index).difference(S_0.index)), list(set(S_0.index).difference(S_1.index)))).items():
    S_1.index = pd.Series(S_1.index).replace(to_replace = tr, value = v, inplace = False)

In [ ]:
mismatch, mm, mm_2 = get_changes(S_1, S_0)

In [222]:
list(mm.keys())

[11, 15, 42, 49, 142, 354, 629, 1368]

In [256]:
m_idx = 1368
mm_2[m_idx]

{'id': 'biomass_protein',
 'reactions': ['60s_maturation',
  'DEUBIQUITINATIONc_PROTEASOMAL_DEGRADATIONc_UBIQUITIN_MONOMER_DEGRADATIONc_1_COMPLEX_DEUBIQUITINATIONc',
  'DEUBIQUITINATIONn_PROTEASOMAL_DEGRADATIONn_COMPLEX_DEUBIQUITINATIONn',
  'TRANSLATION_ELONGATIONc_COMPLEX_DEUBIQUITINATIONc',
  'TRANSLATION_ELONGATIONc_COMPLEX_FORMATIONc',
  'TRANSLATION_ELONGATIONc_COMPLEX_POLYUBIQUITINATIONc',
  'mature_ribosome_COMPLEX_FORMATIONc']}

In [240]:
sln1, stat1, _ = tme1.solve_lp(mu_val = 1e-9)

../../../scripts/core/reaction.py:477 UserWarning: Bounds do not have a mu value


Getting MINOS parameters...
Done in 97.7152 seconds with status 1


In [248]:
tme1.infeasible_reactions(mu_val, sln1, stat1)

{'TRANSCRIPTION_PRE_TRNA_generic': 1.2084017861405103e-18,
 "generic_5'_leader_fragment_tRNA_DEGRADATIONn": 1.2084017861405103e-18,
 "generic_3'_trailer_fragment_tRNA_DEGRADATIONn": 1.2084017861405103e-18,
 'PROCESSING_TRNA_generic': 1.2084017861405103e-18,
 'genericPRIMARY_EXPORTtn': 1.2084017861405103e-18,
 'generic_tRNA_DEGRADATIONc': 1.2084017861405103e-18,
 'SP_degradationr_12_COMPLEX_DEUBIQUITINATIONc': 3.769465066192165e-18}

In [254]:
tme1.reactions.get_by_id('HGNC:10404_TRANSLATION_ELONGATIONc').reaction

'(mu + 0.0227449473351396)/(98855.3094656939*mu + 957.985789455595) HGNC:10404_mrna_c + 0.0227449473351396/(98855.3094656939*mu + 957.985789455595) HGNC:10404_mrna_deg_proxy + 3.5337608067789e6*mu  3.59964221158418e8 TRANSLATION_ELONGATIONc_complex_c + 25 charged_generic_A_trna_c + 5 charged_generic_C_trna_c + 12 charged_generic_D_trna_c + 10 charged_generic_E_trna_c + 11 charged_generic_F_trna_c + 43 charged_generic_G_trna_c + 4 charged_generic_H_trna_c + 17 charged_generic_I_trna_c + 24 charged_generic_K_trna_c + 19 charged_generic_L_trna_c + 7 charged_generic_M_trna_c + 4 charged_generic_N_trna_c + 16 charged_generic_P_trna_c + 6 charged_generic_Q_trna_c + 24 charged_generic_R_trna_c + 14 charged_generic_S_trna_c + 21 charged_generic_T_trna_c + 21 charged_generic_V_trna_c + 3 charged_generic_W_trna_c + 7 charged_generic_Y_trna_c + 293 gtp_c + 294 h2o_c --> HGNC:10404_unfolded_protein_c + 31.3502743800000 biomass_protein + 293 gdp_c + 293 generic_trna_c + 586 h_c + 293 pi_c'

In [250]:
tme1.reactions.get_by_id('generic_tRNA_DEGRADATIONc')

Reaction identifier,generic_tRNA_DEGRADATIONc
Name,
Memory address,0x07f004285e978
Stoichiometry,6.464951664519356e-07 HGNC:30654_enzyme_deg_proxy + 3.52456906415833e5*mu 6.46495166451936e7 HGNC:30654_folded_protein_c + 23.168109272 biomass_tRNA + generic_trna_c + 71 h2o_c --> 13 amp_c + 19 c... 6.464951664519356e-07 + 3.52456906415833e5*mu 6.46495166451936e7 + 23.168109272 + + 71 water --> 13 AMP(2-) + 19 CMP(2-) + 23 GMP(2-) + 72 proton + 17 UMP(2-)
GPR,HGNC:30654
Lower bound,0.0
Upper bound,1000.0


In [251]:
sln1[tme1.reactions.index('generic_tRNA_DEGRADATIONc')]

-1.2084017861405103e-18

In [208]:
S_1.iloc[m_idx, mm[m_idx]['reactions']]

60s_maturation                                                                                          -2.273737e-13
mature_ribosome_COMPLEX_FORMATIONc                                                                       4.547474e-13
TRANSLATION_ELONGATIONc_COMPLEX_POLYUBIQUITINATIONc                                                     -7.206112e-02
TRANSLATION_ELONGATIONc_COMPLEX_DEUBIQUITINATIONc                                                        1.801528e-02
DEUBIQUITINATIONc_PROTEASOMAL_DEGRADATIONc_UBIQUITIN_MONOMER_DEGRADATIONc_1_COMPLEX_DEUBIQUITINATIONc    1.801528e-02
DEUBIQUITINATIONn_PROTEASOMAL_DEGRADATIONn_COMPLEX_DEUBIQUITINATIONn                                     1.801528e-02
TRANSLATION_ELONGATIONc_COMPLEX_FORMATIONc                                                               0.000000e+00
Name: biomass_protein, dtype: float64

In [209]:
S_0.iloc[m_idx, mm[m_idx]['reactions']]

60s_maturation                                                                                           2.273737e-13
mature_ribosome_COMPLEX_FORMATIONc                                                                      -4.547474e-13
TRANSLATION_ELONGATIONc_COMPLEX_POLYUBIQUITINATIONc                                                     -7.206112e-02
TRANSLATION_ELONGATIONc_COMPLEX_DEUBIQUITINATIONc                                                        1.801528e-02
DEUBIQUITINATIONc_PROTEASOMAL_DEGRADATIONc_UBIQUITIN_MONOMER_DEGRADATIONc_1_COMPLEX_DEUBIQUITINATIONc    1.801528e-02
DEUBIQUITINATIONn_PROTEASOMAL_DEGRADATIONn_COMPLEX_DEUBIQUITINATIONn                                     1.801528e-02
TRANSLATION_ELONGATIONc_COMPLEX_FORMATIONc                                                              -2.273737e-13
Name: biomass_protein, dtype: float64

In [213]:
test = S_1.iloc[m_idx, mm[m_idx]['reactions']]  - S_0.iloc[m_idx, mm[m_idx]['reactions']]
test[abs(test) > 1e-12]

Series([], Name: biomass_protein, dtype: float64)

In [214]:
test

60s_maturation                                                                                          -4.547474e-13
mature_ribosome_COMPLEX_FORMATIONc                                                                       9.094947e-13
TRANSLATION_ELONGATIONc_COMPLEX_POLYUBIQUITINATIONc                                                     -2.273737e-13
TRANSLATION_ELONGATIONc_COMPLEX_DEUBIQUITINATIONc                                                        2.273737e-13
DEUBIQUITINATIONc_PROTEASOMAL_DEGRADATIONc_UBIQUITIN_MONOMER_DEGRADATIONc_1_COMPLEX_DEUBIQUITINATIONc    3.552714e-14
DEUBIQUITINATIONn_PROTEASOMAL_DEGRADATIONn_COMPLEX_DEUBIQUITINATIONn                                     3.552714e-14
TRANSLATION_ELONGATIONc_COMPLEX_FORMATIONc                                                               2.273737e-13
Name: biomass_protein, dtype: float64

In [ ]:
# res0 = pd.read_csv(lp_path + 'works_trash.csv', index_col = 0)
# res['og_fluxes'] = res0.loc[res.index.tolist(), :]['reaction_fluxes'].tolist()
# res['diff'] = res['reaction_fluxes'] - res['og_fluxes']

# testing ubiquitin cleavage

In [ ]:
def binary_search(tme, bm_min=-17.111457840000003, bm_max=-0.1, accuracy=0.1):
    feasible_mu = [bm_min]
    infeasible_mu = [bm_max]
    def replace_biomass(biomass_val):
        new_reactions = [r.copy() for r in tme.reactions]
        r_ = [r for r in new_reactions if r.id == 'HGNC:12458_UBIQUITIN_CLEAVAGEc'][0]
        r_.add_metabolites({tme.metabolites.get_by_id('biomass_protein'): biomass_val}, 
                         combine = False)
        if len(r_.check_mass_balance()) == 0:
            test_me = func.ME_Model('test')
            test_me.add_reactions(new_reactions)
            print('Begin solve')
            sln0, stat0, _ = test_me.solve_lp(mu_val = 1e-9)
        if stat0.max() == 0:
            feasible_mu.append(biomass_val)
            return True, sln0, stat0
        elif stat0.max() == 1:
            infeasible_mu.append(biomass_val)
            return False, sln0, stat0
        else:
            raise ValueError('Something went wrong')
    
    while (abs(infeasible_mu[-1] - feasible_mu[-1])) > accuracy:
        print('Current infeasible: {}'.format(infeasible_mu[-1]))
        print('Current feasible: {}'.format(feasible_mu[-1]))
        bool_, sln,stat = replace_biomass((infeasible_mu[-1] + feasible_mu[-1]) * 0.5)
        print('--------------')
    
    return sln, stat, feasible_mu, infeasible_mu
    
    
